## 1. Trim

Session 3 cleaning begins. We load the station-day frame produced in Session 1.5
and confirm it arrived intact before changing anything. This notebook prototypes
each cleaning step on the real data; the proven logic is later assembled into the
Cleaner class in src/data/cleaner.py. We verify the row count, column set, and the
data type of the date column, since later steps filter on the year and that only
works if the date column is a true datetime.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np

from src import config

# the station-day frame produced in Session 1.5
df = pd.read_parquet(config.DAILY_AQI_PATH)
print("loaded:", df.shape)
print(df.dtypes)
df.head()

loaded: (595882, 25)
city                  object
state                 object
station               object
date          datetime64[ns]
pm25                 float64
pm10                 float64
no                   float64
no2                  float64
nox                  float64
nh3                  float64
so2                  float64
co                   float64
o3                   float64
benzene              float64
toluene              float64
xylene               float64
pm25_n                 int64
pm10_n                 int64
so2_n                  int64
no2_n                  int64
nh3_n                  int64
co_n                   int64
o3_n                   int64
aqi                  float64
aqi_bucket            object
dtype: object


,city,state,station,date,pm25,pm10,no,no2,nox,nh3,...,xylene,pm25_n,pm10_n,so2_n,no2_n,nh3_n,co_n,o3_n,aqi,aqi_bucket
0,Agartala,Tripura,TR001,2020-11-08,53.016667,68.795000,1.323333,6.743333,8.565000,4.320000,...,NaN,6,6,6,6,6,6,6,88.200575,Satisfactory
1,Agartala,Tripura,TR001,2020-11-09,42.771304,55.189130,1.668750,6.787083,9.105000,4.706667,...,NaN,23,23,24,24,24,24,24,70.889445,Satisfactory
2,Agartala,Tripura,TR001,2020-11-10,41.183750,54.986250,4.335000,8.927917,15.153333,7.411667,...,NaN,24,24,24,24,24,24,24,68.207026,Satisfactory
3,Agartala,Tripura,TR001,2020-11-11,49.401667,64.949091,1.315833,7.173333,8.950417,4.712083,...,NaN,24,22,24,24,24,24,24,82.092471,Satisfactory
4,Agartala,Tripura,TR001,2020-11-12,45.761250,62.839167,1.043333,7.678333,9.040417,4.743333,...,NaN,24,24,24,24,24,24,24,75.941422,Satisfactory


In [2]:
# Bucket 1a — drop sparse VOC columns
# Banked in EDA: very sparse + weak AQI correlation. Two strikes -> drop.
# (When we assemble cleaner.py, this list moves to config.py as DROP_SPARSE_COLS.)
VOC_COLS = ["benzene", "toluene", "xylene"]

print("cols before:", df.shape[1])
df = df.drop(columns=[c for c in VOC_COLS if c in df.columns])
print("cols after :", df.shape[1])
df.head()

cols before: 25
cols after : 22


,city,state,station,date,pm25,pm10,no,no2,nox,nh3,...,o3,pm25_n,pm10_n,so2_n,no2_n,nh3_n,co_n,o3_n,aqi,aqi_bucket
0,Agartala,Tripura,TR001,2020-11-08,53.016667,68.795000,1.323333,6.743333,8.565000,4.320000,...,1.64000,6,6,6,6,6,6,6,88.200575,Satisfactory
1,Agartala,Tripura,TR001,2020-11-09,42.771304,55.189130,1.668750,6.787083,9.105000,4.706667,...,6.51875,23,23,24,24,24,24,24,70.889445,Satisfactory
2,Agartala,Tripura,TR001,2020-11-10,41.183750,54.986250,4.335000,8.927917,15.153333,7.411667,...,5.01750,24,24,24,24,24,24,24,68.207026,Satisfactory
3,Agartala,Tripura,TR001,2020-11-11,49.401667,64.949091,1.315833,7.173333,8.950417,4.712083,...,6.50875,24,22,24,24,24,24,24,82.092471,Satisfactory
4,Agartala,Tripura,TR001,2020-11-12,45.761250,62.839167,1.043333,7.678333,9.040417,4.743333,...,11.22000,24,24,24,24,24,24,24,75.941422,Satisfactory


### 1b. Clip to the modeling window (2018+)

Early years are a biased sample, not a smaller version of recent ones. The CPCB
station network was small early on and concentrated in heavily polluted locations,
so early averages run artificially high. As the network expanded to cleaner cities,
the average drifted down — composition drift, not a real air-quality trend. Training
on it would teach the model a false downward trend. I restrict modeling to 2018
onward, where row counts stabilize. 2023 is a partial year but kept, as it is the
most recent data and feeds the chronological hold-out test set later.

In [3]:
# year count per row, before clipping — shows the thin early years
print("rows per year (before):")
print(df["date"].dt.year.value_counts().sort_index())

# modeling window start (promoted to config.py when Cleaner is assembled)
MODELING_START_YEAR = 2018

rows_before = len(df)
df = df[df["date"].dt.year >= MODELING_START_YEAR]
rows_after = len(df)

print("\nrows before:", rows_before)
print("rows after :", rows_after)
print("dropped     :", rows_before - rows_after)

rows per year (before):
date
2010      7795
2011     11288
2012     12080
2013     12806
2014     13505
2015     14455
2016     20092
2017     27068
2018     45691
2019     66837
2020     85487
2021    108420
2022    131941
2023     38417
Name: count, dtype: int64

rows before: 595882
rows after : 476793
dropped     : 119089


## 2. Collapse stations into one city-day series

A city-level forecast needs exactly one row per city per day, but 45 of the surviving
cities run multiple stations (Delhi has 39). I collapse them using average
the raw pollutant concentrations across a city's stations for each day, then recompute
the CPCB AQI once on those pooled concentrations. AQI is a nonlinear max-of-sub-indices
rule, so the AQI of the averaged concentrations is the correct city value, not the
average of the per-station AQIs. The stale per-station aqi, aqi_bucket, and valid-hour
count columns are dropped and the AQI is rebuilt fresh on the pooled data.

In [4]:
# pollutant concentrations to pool across a city's stations
# (VOCs already dropped; these 9 remain — the 7 CPCB pollutants plus no/nox)
POLLUTANT_COLS = ["pm25", "pm10", "no", "no2", "nox", "nh3", "so2", "co", "o3"]

stations_before = df.groupby("city")["station"].nunique().max()
rows_before = len(df)

# average concentrations across stations -> one row per city-day.
# selecting only POLLUTANT_COLS here drops the stale per-station aqi/bucket/_n cols.
city_day = (
    df.groupby(["city", "state", "date"], as_index=False)[POLLUTANT_COLS]
    .mean()
)

# recompute CPCB AQI + category once on the pooled concentrations
from src.data.aqi import AQICalculator

calc = AQICalculator()
city_day = calc.compute(city_day)   # returns df with 'aqi' and 'aqi_bucket' added

print("rows before (station-day):", rows_before)
print("rows after  (city-day)   :", len(city_day))
print("max stations in one city (gone now):", stations_before)
print("unique city-day rows == row count:",
      len(city_day) == len(city_day[["city", "date"]].drop_duplicates()))
print("\ncolumns now:", city_day.columns.tolist())
city_day.head()

rows before (station-day): 476793
rows after  (city-day)   : 255161
max stations in one city (gone now): 40
unique city-day rows == row count: False

columns now: ['city', 'state', 'date', 'pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'so2', 'co', 'o3', 'aqi', 'aqi_bucket']


,city,state,date,pm25,pm10,no,no2,nox,nh3,so2,co,o3,aqi,aqi_bucket
0,Agartala,Tripura,2020-11-08,53.016667,68.795000,1.323333,6.743333,8.565000,4.320000,11.036667,0.651667,1.64000,88.200575,Satisfactory
1,Agartala,Tripura,2020-11-09,42.771304,55.189130,1.668750,6.787083,9.105000,4.706667,11.220833,0.858750,6.51875,70.889445,Satisfactory
2,Agartala,Tripura,2020-11-10,41.183750,54.986250,4.335000,8.927917,15.153333,7.411667,11.320000,1.085000,5.01750,68.207026,Satisfactory
3,Agartala,Tripura,2020-11-11,49.401667,64.949091,1.315833,7.173333,8.950417,4.712083,11.314167,1.108750,6.50875,82.092471,Satisfactory
4,Agartala,Tripura,2020-11-12,45.761250,62.839167,1.043333,7.678333,9.040417,4.743333,11.291667,0.827500,11.22000,75.941422,Satisfactory


### 3a. Diagnose duplicate city-day rows

The collapse left some city-date pairs with more than one row, which would corrupt
per-city lag features later. The group key included state, so any city name that
appears under two different state labels produces two rows per date. I inspect the
colliding city/state pairs to decide whether these are genuinely different cities
sharing a name, or one city with inconsistent state labels.

In [5]:
# flag every row sharing a city-date with another (keep=False marks all copies)
dupe_mask = city_day.duplicated(subset=["city", "date"], keep=False)
dupes = city_day[dupe_mask]

print("duplicate city-day rows :", len(dupes))
print("cities involved         :", dupes["city"].nunique())

# the distinct city/state pairs caught in collisions — the root cause
combos = (
    dupes[["city", "state"]]
    .drop_duplicates()
    .sort_values(["city", "state"])
    .reset_index(drop=True)
)
print("\ncity/state pairs in collisions:")
print(combos)

duplicate city-day rows : 1008
cities involved         : 1

city/state pairs in collisions:
         city        state
0  Aurangabad        Bihar
1  Aurangabad  Maharashtra


### 3b. Lock composite city identity (city + state)

The collapse was correct, but the uniqueness check was wrong: it tested city + date
only, while a city's true identity is city + state. Aurangabad exists in both Bihar
and Maharashtra — two real, different cities sharing a name — so checking on city
alone falsely flagged them as duplicates. I create a single city_id column combining
city and state, which becomes the identity key for all downstream per-city work
(lag features, the city selector). Grouping on city_id makes it impossible to merge
two same-named cities by accident.

In [6]:
# single identity key: city + state (so same-named cities stay distinct)
city_day["city_id"] = city_day["city"] + ", " + city_day["state"]

# re-run the uniqueness check on the CORRECT key
is_unique = len(city_day) == len(city_day[["city_id", "date"]].drop_duplicates())
print("unique city_id-day rows == row count:", is_unique)

# true distinct-city count (Aurangabad now splits into two)
print("distinct city names :", city_day["city"].nunique())
print("distinct city_ids   :", city_day["city_id"].nunique())

# confirm the Aurangabad split is clean
print("\nAurangabad ids:")
print(city_day[city_day["city"] == "Aurangabad"]["city_id"].unique())

unique city_id-day rows == row count: True
distinct city names : 241
distinct city_ids   : 242

Aurangabad ids:
['Aurangabad, Bihar' 'Aurangabad, Maharashtra']


## 4. Drop label-missing rows, filter thin cities, recheck nh3

Three row-level filters in sequence, ordered so each sees the result of the last.
First I drop rows with no AQI value: a row without the target is useless for
forecasting. Then I count labelled days per city_id and keep only cities with at
least MIN_CITY_DAYS (730 = two seasonal cycles), which lag and rolling features
need. The survivor count is computed fresh here on the post-2018, collapsed,
labelled frame, so it will differ from the EDA estimate. Finally I recheck nh3
missingness on the survivors, since the thin cities just dropped may have been its
worst offenders — I measure before deciding whether to keep or drop the column.

In [7]:
# MIN_CITY_DAYS promoted to config.py when Cleaner is assembled
MIN_CITY_DAYS = 730

# --- step 1: drop rows with no AQI label ---
rows_before = len(city_day)
city_day = city_day[city_day["aqi"].notna()]
print("step 1 — drop label-missing rows")
print("  rows before:", rows_before)
print("  rows after :", len(city_day))
print("  dropped    :", rows_before - len(city_day))

# --- step 2: keep only cities with enough labelled days ---
# transform broadcasts each city's day-count back onto its rows -> full-length mask
day_counts = city_day.groupby("city_id")["date"].transform("size")
cities_before = city_day["city_id"].nunique()
city_day = city_day[day_counts >= MIN_CITY_DAYS]
print("\nstep 2 — filter cities with <", MIN_CITY_DAYS, "labelled days")
print("  city_ids before:", cities_before)
print("  city_ids after :", city_day["city_id"].nunique())
print("  rows after     :", len(city_day))

# --- step 3: recheck nh3 missingness on the survivors ---
nh3_missing_pct = city_day["nh3"].isna().mean() * 100
print("\nstep 3 — nh3 missingness on survivors")
print("  nh3 missing: {:.1f}%".format(nh3_missing_pct))
print("  (was 37.8% pre-filter)")

step 1 — drop label-missing rows
  rows before: 255161
  rows after : 239966
  dropped    : 15195

step 2 — filter cities with < 730 labelled days
  city_ids before: 241
  city_ids after : 141
  rows after     : 214976

step 3 — nh3 missingness on survivors
  nh3 missing: 17.4%
  (was 37.8% pre-filter)


## 5. Time-aware fill of short predictor gaps

The pollutant columns still contain gaps that would propagate into lag and rolling
features. I fill them in a leakage-safe, time-aware way rather than with a column
mean. Filling is done per city_id, in date order, using forward-fill so only past
readings ever fill a later gap — never future values. The fill is capped at
FILL_LIMIT days so short sensor outages are bridged while long silences are left as
NaN rather than fabricated. A column mean would ignore time and smear one city's
average across the gap; forward-fill respects that air quality is a smooth daily
process where the last known reading is the best estimate for a short gap.

In [8]:
# fill config (promoted to config.py when Cleaner is assembled)
FILL_LIMIT = 3   # bridge gaps up to 3 consecutive days; leave longer ones as NaN
POLLUTANT_COLS = ["pm25", "pm10", "no", "no2", "nox", "nh3", "so2", "co", "o3"]

# date order WITHIN each city is required for ffill to mean "carry the past forward"
city_day = city_day.sort_values(["city_id", "date"]).reset_index(drop=True)

# missingness before, per pollutant
missing_before = city_day[POLLUTANT_COLS].isna().mean().mul(100).round(1)

# forward-fill per city, capped — groupby stops one city leaking into the next
city_day[POLLUTANT_COLS] = (
    city_day.groupby("city_id")[POLLUTANT_COLS].ffill(limit=FILL_LIMIT)
)

# missingness after
missing_after = city_day[POLLUTANT_COLS].isna().mean().mul(100).round(1)

comparison = pd.DataFrame({"before_%": missing_before, "after_%": missing_after})
comparison["filled_pts"] = (comparison["before_%"] - comparison["after_%"]).round(1)
print("per-pollutant missingness, before vs after capped ffill:")
print(comparison)
print("\ntotal rows:", len(city_day))

per-pollutant missingness, before vs after capped ffill:
      before_%  after_%  filled_pts
pm25       3.5      3.1         0.4
pm10       6.5      6.1         0.4
no         8.4      7.9         0.5
no2        2.9      2.4         0.5
nox        9.3      9.0         0.3
nh3       17.4     16.8         0.6
so2        3.1      2.7         0.4
co         7.0      6.2         0.8
o3         8.5      7.7         0.8

total rows: 214976


### 5b. Diagnose gap structure before trusting the fill

The capped forward-fill reclaimed very little, which is suspicious for daily sensor
data. Forward-fill can only fill a NaN inside a row that exists; it cannot recreate
entirely missing calendar days. If cities are missing whole dates from their row
grid, the fill limit measured in rows no longer equals a limit in days, breaking the
leakage-safe reasoning. I inspect one well-covered city to see whether gaps are
within-row NaNs (fillable) or missing dates (needs a complete daily grid first).

In [9]:
# pick one well-covered survivor city to inspect
sample_id = city_day["city_id"].value_counts().index[0]
sample = city_day[city_day["city_id"] == sample_id].sort_values("date")
print("inspecting:", sample_id, "| rows:", len(sample))

# 1) missing WHOLE DAYS: compare actual dates to the complete expected range
full_range = pd.date_range(sample["date"].min(), sample["date"].max(), freq="D")
actual_days = sample["date"].nunique()
expected_days = len(full_range)
print("\nmissing whole days (no row at all):")
print("  expected days in span:", expected_days)
print("  actual days present  :", actual_days)
print("  missing dates        :", expected_days - actual_days)

# 2) are within-row NaN runs even longer than our limit of 3?
#    measure consecutive-NaN run lengths for one pollutant on this city
is_na = sample["nh3"].isna()
run_id = (~is_na).cumsum()                      # new id each time a value appears
run_lengths = is_na.groupby(run_id).sum()       # length of each NaN run
runs = run_lengths[run_lengths > 0]
print("\nnh3 consecutive-NaN run lengths on this city:")
print("  number of gaps     :", len(runs))
print("  longest gap (rows) :", int(runs.max()) if len(runs) else 0)
print("  gaps longer than 3 :", int((runs > 3).sum()))

inspecting: Noida, Uttar Pradesh | rows: 1916

missing whole days (no row at all):
  expected days in span: 1916
  actual days present  : 1916
  missing dates        : 0

nh3 consecutive-NaN run lengths on this city:
  number of gaps     : 0
  longest gap (rows) : 0
  gaps longer than 3 : 0


In [10]:
# confirm complete daily grids across ALL survivor cities, not just the sample
grid_check = (
    city_day.groupby("city_id")["date"]
    .agg(actual="nunique", first="min", last="max")
)
grid_check["expected"] = (grid_check["last"] - grid_check["first"]).dt.days + 1
grid_check["missing_days"] = grid_check["expected"] - grid_check["actual"]

print("cities with a complete daily grid (0 missing days):",
      (grid_check["missing_days"] == 0).sum(), "/", len(grid_check))
print("cities WITH missing dates                         :",
      (grid_check["missing_days"] > 0).sum())
print("\nworst 5 cities by missing days:")
print(grid_check.sort_values("missing_days", ascending=False).head(5))

cities with a complete daily grid (0 missing days): 9 / 141
cities WITH missing dates                         : 132

worst 5 cities by missing days:
                           actual      first       last  expected  \
city_id                                                             
Jorapokhar, Jharkhand        1202 2018-01-01 2023-03-31      1916   
Durgapur, West Bengal        1318 2018-01-01 2023-03-31      1916   
Kolar, Karnataka             1189 2018-06-19 2023-03-31      1747   
Chamarajanagar, Karnataka     900 2019-07-19 2023-03-31      1352   
Haldia, West Bengal          1501 2018-01-01 2023-03-31      1916   

                           missing_days  
city_id                                  
Jorapokhar, Jharkhand               714  
Durgapur, West Bengal               598  
Kolar, Karnataka                    558  
Chamarajanagar, Karnataka           452  
Haldia, West Bengal                 415  


### 5c. Reindex each city to a complete daily grid, then fill

The capped forward-fill in 5 reclaimed almost nothing because most cities are
missing whole calendar dates, not just scattered values inside existing rows.
Forward-fill cannot recreate a day that has no row, and with date-sparse rows my
3-row cap no longer means a 3-day cap — which quietly breaks the leakage-safe
reasoning behind the fill. So before any fill, I rebuild each city's row grid: for
every city_id I generate the full daily date range from its first to last day and
insert a blank (NaN) row for every missing date. On this complete grid the previous
row is always literally yesterday, so lag and rolling features in Session 4 are
honest by construction. I then redo the forward-fill on the 9 predictor pollutants
only — never on aqi, which is the label and must stay tied to real measurements. The
fill is capped at FILL_LIMIT days, which now genuinely means days because the grid
has no holes. Long gaps are left as NaN rather than fabricated. This step supersedes
the standalone fill in 5.

In [11]:
# grid + fill config (promoted to config.py when Cleaner is assembled)
FILL_LIMIT = 3   # bridge gaps up to 3 consecutive DAYS; leave longer ones as NaN
POLLUTANT_COLS = ["pm25", "pm10", "no", "no2", "nox", "nh3", "so2", "co", "o3"]
IDENTITY_COLS = ["city_id", "city", "state"]


def build_daily_grid(frame: pd.DataFrame) -> pd.DataFrame:
    """Reindex every city to a gap-free daily date range.

    For each city_id, inserts a NaN row for every calendar day missing between
    its first and last day, so 'previous row' always means 'yesterday' for the
    lag features built later.
    """
    pieces = []
    for city_id, group in frame.groupby("city_id"):
        full_range = pd.date_range(
            group["date"].min(), group["date"].max(), freq="D"
        )
        gridded = group.set_index("date").reindex(full_range)
        gridded.index.name = "date"

        # reindex blanks the identity columns on inserted rows; restore them
        identity = group.iloc[0]
        for col in IDENTITY_COLS:
            gridded[col] = identity[col]

        pieces.append(gridded.reset_index())
    return pd.concat(pieces, ignore_index=True)


# --- step 1: reindex every city to its complete daily grid ---
rows_before = len(city_day)
city_day = build_daily_grid(city_day)
print("step 1 — reindex to complete daily grid")
print("  rows before:", rows_before)
print("  rows after :", len(city_day))
print("  inserted   :", len(city_day) - rows_before, "blank daily rows")

# --- step 2: redo the capped forward-fill on the gridded frame ---
# date order WITHIN each city is required for ffill to carry the past forward
city_day = city_day.sort_values(["city_id", "date"]).reset_index(drop=True)

missing_before = city_day[POLLUTANT_COLS].isna().mean().mul(100).round(1)

# fill ONLY the predictor pollutants, capped — never the aqi label
city_day[POLLUTANT_COLS] = (
    city_day.groupby("city_id")[POLLUTANT_COLS].ffill(limit=FILL_LIMIT)
)

missing_after = city_day[POLLUTANT_COLS].isna().mean().mul(100).round(1)

comparison = pd.DataFrame({"before_%": missing_before, "after_%": missing_after})
comparison["filled_pts"] = (comparison["before_%"] - comparison["after_%"]).round(1)
print("\nstep 2 — per-pollutant missingness on the GRIDDED frame, before vs after ffill:")
print(comparison)

step 1 — reindex to complete daily grid
  rows before: 214976
  rows after : 224808
  inserted   : 9832 blank daily rows

step 2 — per-pollutant missingness on the GRIDDED frame, before vs after ffill:
      before_%  after_%  filled_pts
pm25       7.4      5.9         1.5
pm10      10.2      8.6         1.6
no        11.9     10.3         1.6
no2        6.7      5.0         1.7
nox       13.0     11.6         1.4
nh3       20.4     19.1         1.3
so2        7.0      5.3         1.7
co        10.3      8.7         1.6
o3        11.7     10.2         1.5


In [12]:
# every city's distinct-day count should now equal its full expected span
spans = city_day.groupby("city_id")["date"].agg(["min", "max", "nunique"])
spans["expected"] = (spans["max"] - spans["min"]).dt.days + 1
spans["missing"]  = spans["expected"] - spans["nunique"]

print("cities still missing any dates:", (spans["missing"] > 0).sum(), "/", len(spans))
print("max missing days for any city :", int(spans["missing"].max()))

cities still missing any dates: 0 / 141
max missing days for any city : 0


In [13]:
city_day.columns.to_list()

['date',
 'city',
 'state',
 'pm25',
 'pm10',
 'no',
 'no2',
 'nox',
 'nh3',
 'so2',
 'co',
 'o3',
 'aqi',
 'aqi_bucket',
 'city_id']

## 6. Crystallize the cleaning pipeline into src/data/cleaner.py

Every cleaning step above is now proven on the bench, so I assemble them into a
single Cleaner class — the importable, deterministic production version that
Session 4's FeatureBuilder and the app will call. I run it end to end on the raw
daily_aqi.parquet, confirm it reproduces the numbers I saw step by step (gap-free
grid, ~224.8k rows, nh3 honest at ~19%), and persist the result to
clean_city_day.parquet. That file is the single input the rest of the project sees.

In [14]:
from src.data.cleaner import Cleaner

clean = Cleaner().clean()

print("shape          :", clean.shape)
print("city_ids       :", clean["city_id"].nunique())
print("columns        :", clean.columns.tolist())
print("nh3 missing %  :", round(clean["nh3"].isna().mean() * 100, 1))

# re-confirm the gap-free grid the asserts also check
spans = clean.groupby("city_id")["date"].agg(["min", "max", "nunique"])
missing = (spans["max"] - spans["min"]).dt.days + 1 - spans["nunique"]
print("cities missing dates:", (missing > 0).sum(), "/", len(spans))

# persist — this parquet is the only input Session 4 reads
config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
clean.to_parquet(config.CLEAN_CITY_DAY_PATH, index=False)
print("saved ->", config.CLEAN_CITY_DAY_PATH)

Task was destroyed but it is pending!
task: <Task pending name='Task-86' coro=<_async_in_context.<locals>.run_in_context() done, defined at D:\07_Data_Science\05_DS_Projects\03_AirCast\.venv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-87' coro=<Kernel.shell_main() running at D:\07_Data_Science\05_DS_Projects\03_AirCast\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at D:\07_Data_Science\05_DS_Projects\03_AirCast\.venv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>
C:\Users\mangl\AppData\Local\Programs\Python\Python313\Lib\ast.py:54: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  return compile(source, filename, mode, flags,
Task was destroyed but it is pending!
task: <Task pending name='Task-87' coro=<Kernel.shell_main() running at D:\07_Data_Science\05_DS_Projects\03_AirCast\.venv\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]>


shape          : (224808, 15)
city_ids       : 141
columns        : ['city_id', 'city', 'state', 'date', 'pm25', 'pm10', 'no', 'no2', 'nox', 'nh3', 'so2', 'co', 'o3', 'aqi', 'aqi_bucket']
nh3 missing %  : 19.4
cities missing dates: 0 / 141
saved -> D:\07_Data_Science\05_DS_Projects\03_AirCast\data\processed\clean_city_day.parquet
